In [ ]:
import pandas as pd
import numpy as np

champs_lookup = pd.read_csv("../data/raw/champs.csv")
matches = pd.read_csv("../data/raw/matches.csv")
participants = pd.read_csv("../data/raw/participants.csv")
stats1 = pd.read_csv("../data/raw/stats1.csv")
stats2 = pd.read_csv("../data/raw/stats2.csv",low_memory=False)
teambans = pd.read_csv("../data/raw/teambans.csv")

In [ ]:
#only euw matches checked, dropped ~10k rows
matches = matches[matches['platformid']=='EUW1']
lookup_ids = list(matches['id'].copy())

champ_names = champs_lookup.set_index("id")["name"]

position = participants["role"].map({"DUO_SUPPORT": "support", "DUO_CARRY": "adc"})
not_bot = participants["position"] != "BOT"
position = position.where(~not_bot, participants["position"].str.lower())

player_lookup = pd.DataFrame({
    "player_id":    participants["id"],
    "match_id":     participants["matchid"],
    "team":         np.where(participants["player"] < 6, "blue", "red"),
    "ss1":          participants["ss1"],
    "ss2":          participants["ss2"],
    "position":     position,
    "played_champ": participants["championid"].map(champ_names),
})

before = len(player_lookup)
player_lookup = (
    player_lookup[player_lookup["match_id"].isin(lookup_ids)]
    .reset_index(drop=True)
)
print(f"kept {len(player_lookup):,} of {before:,} rows "
      f"({len(player_lookup) / before:.1%})")
player_lookup

In [ ]:
# somehow grasp scaling champs and non-scaling ones
# pregame knowledge focus, champion winrates, expected kdas, expected dmg types and amounts, expected tanking types and amounts, avg objdmgs, avg goldspent, maybe how many items the champ buys
# might need outside source scraping u.gg or smth
stats = pd.concat([stats1, stats2], ignore_index=True)
del stats1, stats2
stats

In [ ]:
ITEM_COLS = [f"item{i}" for i in range(1, 7)]

AVG_COLS = {
    "kills": "kills", "deaths": "deaths", "assists": "assists",
    "totdmgdealt": "dmg_total", "magicdmgdealt": "dmg_magic",
    "physicaldmgdealt": "dmg_phys", "truedmgdealt": "dmg_true",
    "totdmgtochamp": "champ_dmg_total", "magicdmgtochamp": "champ_dmg_magic",
    "physdmgtochamp": "champ_dmg_phys", "truedmgtochamp": "champ_dmg_true",
    "totheal": "heal",
    "dmgtoobj": "dmg_obj", "dmgtoturrets": "dmg_turret",
    "totdmgtaken": "taken_total", "magicdmgtaken": "taken_magic",
    "physdmgtaken": "taken_phys", "truedmgtaken": "taken_true",
    "goldspent": "gold_spent",
    "champlvl": "champ_lvl",
    "totcctimedealt": "cc_time",
    "wardsplaced": "wards_placed", "wardskilled": "wards_killed",
}


def build_champion_meta(player_lookup, stats, matches, match_ids=None, prior_games=100):
    pl = player_lookup
    if match_ids is not None:
        pl = pl[pl["match_id"].isin(match_ids)]

    df = (
        pl[["player_id", "match_id", "played_champ", "position"]]
        .merge(stats[["id", "win", *ITEM_COLS, *AVG_COLS]],
               left_on="player_id", right_on="id", how="inner")
        .merge(matches[["id", "duration"]].rename(columns={"id": "match_id"}),
               on="match_id", how="left")
        .dropna(subset=["played_champ"])
    )
    df["win"] = df["win"].astype(float)
    g = df.groupby("played_champ")

    #rarer champs handled with smoothing
    meta = pd.DataFrame({"games": g.size()})
    meta["pick_rate"] = meta["games"] / df["match_id"].nunique()
    meta["winrate"] = g["win"].mean()
    meta["winrate_smooth"] = (
        (g["win"].sum() + prior_games * df["win"].mean()) / (meta["games"] + prior_games)
    )

    meta = meta.join(g[list(AVG_COLS)].mean().rename(columns=AVG_COLS))

    #kda needs spec cuz 0 death
    tot = g[["kills", "deaths", "assists"]].sum()
    meta["kda"] = (tot["kills"] + tot["assists"]) / tot["deaths"].clip(lower=1)

    for kind in ("magic", "phys", "true"):
        meta[f"champ_dmg_{kind}_share"] = meta[f"champ_dmg_{kind}"] / meta["champ_dmg_total"]
    meta["gold_per_kill"] = meta["gold_spent"] / meta["kills"].replace(0, np.nan)

    filled = df[ITEM_COLS].fillna(0).ne(0)
    filled.columns = [f"has_item_{i}" for i in range(1, 7)]
    meta = meta.join(filled.groupby(df["played_champ"]).mean())

    #is item in slot n
    meta["items_filled"] = meta[[f"has_item_{i}" for i in range(1, 7)]].sum(axis=1)

    pos = pd.crosstab(df["played_champ"], df["position"], normalize="index")
    meta = meta.join(pos.add_prefix("pos_"))

    # scaling: winrate in the shortest vs longest third of games, might need a better approach
    length = pd.qcut(df["duration"], 3, labels=["short", "mid", "long"])
    by_len = (df.groupby(["played_champ", length], observed=True)["win"]
                .mean().unstack().add_prefix("winrate_"))
    meta = meta.join(by_len)
    meta["scaling"] = meta["winrate_long"] - meta["winrate_short"]
    return meta

In [ ]:
order = matches.sort_values("creation")["id"]
cut = int(len(order) * 0.8)
train_ids, test_ids = set(order[:cut]), set(order[cut:])

# for this setup, need to split to not leak data
champion_meta = build_champion_meta(player_lookup, stats, matches, match_ids=train_ids)

In [ ]:
#win -> winrate
#item1-6 -> has_item_n
#kills, deaths, assists -> kda
#totdmgdealt, magicdmgdealt, physicaldmgdealt, truedmgdealt -> respective total dmgs
#totdmgtochamp, magicdmgtochamp, physdmgtochamp, truedmgtochamp -> respective dmgs in fights
#totheal -> heal
#dmgtoobj, dmgtoturrets -> obj + turret dam
#totdmgtaken, magicdmgtaken, physdmgtaken, truedmgtaken -> tank stats
#goldspent -> usage of gold + testing baus theory of goldspent/kills lol
#champlvl -> kind of shows scaling, need extensive check
#totcctimedealt -> cc
#wardsplaced, wardskilled -> vision, maybe useful